# 06 – Response Synthesis

In [1]:
# config: store, retrieval, intent, clause analyzer, risk, response composer, and query
import sys
from pathlib import Path

ROOT = Path("../").resolve()
sys.path.insert(0, str(ROOT))

from rag.config import get_settings
from rag.retrieval.store import IndexStore
from rag.retrieval.hybrid_search import HybridRetrieval
from rag.agents.intent import IntentAgent
from rag.agents.clause_analyzer import ClauseAnalyzerAgent
from rag.agents.response_composer import ResponseComposerAgent

settings = get_settings()
store = IndexStore(persist_directory=ROOT / settings.persist_directory)
retrieval = HybridRetrieval(store=store, top_k=5)
intent_agent = IntentAgent()
clause_agent = ClauseAnalyzerAgent()
composer = ResponseComposerAgent()

QUERY = "What is the notice period for terminating the NDA and does confidentiality survive?"

In [2]:
# run pipeline: intent (rule) → retrieve → extract clauses (1 LLM) → risk (rule) → compose (rule); print answer, citations, risks
from rag.agents.risk_rule_engine import evaluate_risk

intent = intent_agent.classify(QUERY)
retrieved = retrieval.search(intent.refined_query or QUERY)
extracted = clause_agent.analyze(retrieved)
risk = evaluate_risk(extracted, retrieved, intent_type=intent.intent_type)
response = composer.compose(QUERY, intent, retrieved, extracted, risk)

print("Direct answer:", response.direct_answer)
print("Cited clauses:", response.cited_clauses)
print("Risk flags:", response.risk_flags)
print("Follow-up:", response.follow_up_suggestions)
print("Insufficient info:", response.insufficient_info)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Direct answer: {
"direct_answer": "The notice period for terminating the NDA is thirty (30) days written notice. Confidentiality obligations survive termination for five (5) years.",
"cited_clauses": [
{"document": "Nda Acme Vendor", "section": "3", "title": "Term and Termination", "excerpt": "Either party may terminate this Agreement with thirty (30) days written notice."},
{"document": "Nda Acme Vendor", "section": "", "title": "Confidentiality Obligations", "excerpt": "Confidentiality obligations shall survive termination for five (5) years."}
],
"risk_flags": [
{"level": "LOW", "message": "No high-risk clauses detected"}
],
"follow_up_suggestions": [
"What are the consequences of disclosing confidential information during the notice period?",
"How does the NDA define 'Confidential Information'?"
]
"insufficient_info": false
}
Cited clauses: []
Risk flags: []
Follow-up: []
Insufficient info: True
